# 22 — Attacchi presenti e assenti nel training (T38)

Confrontiamo le predizioni già salvate sui 22.544 casi del test ufficiale.
Un attacco è "assente dal training" se il suo **nome specifico** non compare in
KDDTrain+. Le cinque famiglie restano quelle già note ai modelli.

Normal viene tenuto separato: includerlo tra gli attacchi già presenti renderebbe
il confronto poco leggibile. Questa è una diagnosi descrittiva, non un nuovo training
e non una misura operativa di rilevamento zero-day.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, f1_score, accuracy_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES
from src.targets import AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER
OUTPUT = ROOT / 'artifacts/seen_unseen_attacks'
OUTPUT.mkdir(parents=True, exist_ok=True)
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
for name, digest in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES)
assert test['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()
expected_truth = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).to_numpy(dtype=str)
models = {
    'decision_tree': ('Decision Tree', 'decision_tree_official_test'),
    'random_forest_balanced': ('Random Forest', 'random_forest_balanced_official_test'),
    'gaussian_naive_bayes': ('Gaussian Naive Bayes', 'gaussian_naive_bayes_official_test'),
    'svm_rbf': ('SVM RBF', 'svm_rbf_official_test'),
    'neural_network': ('Rete neurale', 'neural_network_official_test'),
    'kmeans': ('K-Means', 'kmeans_official_test'),
}
train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES)
assert train['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()

## 1. Verificare le predizioni salvate

Stesso protocollo, parametri, input, target e indici dei sei esperimenti ufficiali. Ricalcoliamo le confusion matrix per verificare la coerenza degli artefatti. Nessun fit o nuova predizione.

In [ ]:
prediction_frames = []
overall_rows = []
class_rows = []
matrices = {}
saved_results = {}
for model_id, (label, artifact) in models.items():
    folder = ROOT / 'artifacts' / artifact
    result = json.loads((folder / 'results.json').read_text(encoding='utf-8'))
    assert result['protocol_sha256'] == protocol_hash
    assert result['protocol_id'] == protocol['protocol_id']
    assert result['input_sha256'] == protocol['input_sha256']
    assert result['label_profile'] == protocol['label_profile']
    assert result['family_order'] == list(FAMILY_ORDER)
    assert result['train_rows'] == 125973 and result['test_rows'] == len(expected_truth)
    assert result['classifier_parameters'] == protocol['models'][model_id]['parameters']
    assert hashlib.sha256((folder / 'predictions.npz').read_bytes()).hexdigest() == result['predictions_sha256']
    with np.load(folder / 'predictions.npz', allow_pickle=False) as saved:
        truth, predicted = saved['truth'], saved['predicted']
        np.testing.assert_array_equal(saved['test_indices'], np.arange(len(expected_truth)))
    np.testing.assert_array_equal(truth, expected_truth)
    assert set(predicted) <= set(FAMILY_ORDER)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    normalized = confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true')
    np.testing.assert_array_equal(counts, result['confusion_counts'])
    np.testing.assert_allclose(normalized, result['confusion_normalized'])
    precision, recall, f1, support = precision_recall_fscore_support(truth, predicted, labels=FAMILY_ORDER, zero_division=0)
    scores = {
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
        'accuracy': accuracy_score(truth, predicted),
    }
    for metric, value in scores.items():
        np.testing.assert_allclose(value, result['metrics'][metric])
    overall_rows.append({'model_id': model_id, 'model': label, **scores, 'fit_seconds': result['fit_seconds']})
    for i, family in enumerate(FAMILY_ORDER):
        class_rows.append({'model_id': model_id, 'model': label, 'family': family,
                           'precision': precision[i], 'recall': recall[i], 'f1': f1[i],
                           'support': int(support[i]), 'true_positives': int(counts[i,i]),
                           'false_positives': int(counts[:,i].sum()-counts[i,i]),
                           'error_rate': 1-recall[i]})
    prediction_frames.append(pd.DataFrame({
        'model_id': model_id, 'model': label, 'test_index': np.arange(len(truth)),
        'attack': test['attack_label'].to_numpy(), 'family': truth, 'predicted': predicted,
    }))
    matrices[model_id] = normalized
    saved_results[model_id] = result
overall = pd.DataFrame(overall_rows)
per_class = pd.DataFrame(class_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

## 2. Formare i gruppi dal vocabolario del training

La presenza riguarda il tipo di attacco, non la presenza della singola connessione.
I dati normali restano un terzo gruppo. L'inventario comprende anche attacchi
presenti nel training ma assenti nel test.

In [ ]:
training_attacks = set(train.loc[train['attack_label'].ne('normal'), 'attack_label'])
groups = pd.Series(np.where(
    test['attack_label'].eq('normal'), 'Normal',
    np.where(test['attack_label'].isin(training_attacks),
             'Attacchi presenti nel training', 'Attacchi assenti dal training'),
), index=test.index, name='group')

In [ ]:
inventory = pd.DataFrame({
    'train_support': train['attack_label'].value_counts(),
    'test_support': test['attack_label'].value_counts(),
}).fillna(0).astype(int).rename_axis('attack').reset_index()
inventory['family'] = inventory['attack'].map(AUTHOR_ATTACK_TO_FAMILY)
inventory['present_in_train'] = inventory['train_support'].gt(0)
inventory['present_in_test'] = inventory['test_support'].gt(0)
assert inventory['family'].notna().all()
assert inventory['train_support'].sum() == len(train)
assert inventory['test_support'].sum() == len(test)
predictions['group'] = groups.to_numpy()[predictions['test_index'].to_numpy()]
composition = pd.crosstab(groups, pd.Series(expected_truth, name='family')).reindex(columns=FAMILY_ORDER, fill_value=0)
display(composition)
display(inventory.sort_values(['family','test_support'], ascending=[True,False]))

## 3. Riconoscimento della famiglia e rilevamento dell'attacco

**Famiglia corretta**: frazione del gruppo assegnata alla famiglia vera.
**Attacco rilevato**: frazione degli attacchi assegnata a una qualunque famiglia
di attacco, anche sbagliata. Sono due denominatori uguali, ma criteri diversi.
Il secondo valore è recall di rilevamento sui soli attacchi; non misura i falsi
allarmi. Per Normal viene lasciato vuoto, mentre la correttezza coincide con
la quota di connessioni normali riconosciute come tali.

In [ ]:
diagnosis = predictions.assign(
    family_correct=predictions['family'].eq(predictions['predicted']),
    detected_as_attack=predictions['predicted'].ne('Normal'),
)
group_summary = diagnosis.groupby(['model','group'], sort=False).agg(
    support=('family','size'), family_correct=('family_correct','sum'),
    detected_as_attack=('detected_as_attack','sum'),
).reset_index()
group_summary['family_recall'] = group_summary['family_correct'] / group_summary['support']
group_summary['attack_detection_recall'] = group_summary['detected_as_attack'] / group_summary['support']
group_summary.loc[group_summary['group'].eq('Normal'), 'attack_detection_recall'] = np.nan
family_summary = diagnosis.groupby(['model','group','family'], sort=False).agg(
    support=('family','size'), family_correct=('family_correct','sum'),
    detected_as_attack=('detected_as_attack','sum'),
).reset_index()
family_summary['family_recall'] = family_summary['family_correct'] / family_summary['support']
family_summary['attack_detection_recall'] = family_summary['detected_as_attack'] / family_summary['support']
family_summary.loc[family_summary['family'].eq('Normal'), 'attack_detection_recall'] = np.nan

In [ ]:
display(group_summary.round(4))
display(family_summary[family_summary['family'].ne('Normal')].round(4))

## 4. Confronto grafico

Le percentuali aggregate pesano ogni riga del test allo stesso modo. I due gruppi
hanno composizioni diverse: per questo affianchiamo il confronto entro famiglia.
Un divario non dimostra da solo che la causa sia la mancata presenza nel training.

In [ ]:
attack_groups = ['Attacchi presenti nel training','Attacchi assenti dal training']
labels = [label for label, _ in models.values()]
short_labels = ['DT','RF','GaussianNB','SVM','MLP','K-Means']
fig, axes = plt.subplots(1,2,figsize=(13,5),layout='constrained')
for axis, metric, title in zip(axes, ['family_recall','attack_detection_recall'], ['Famiglia corretta','Rilevato come attacco']):
    values = group_summary[group_summary['group'].isin(attack_groups)].pivot(index='model',columns='group',values=metric).reindex(labels)
    for j, group in enumerate(attack_groups):
        bars = axis.bar(np.arange(6)+(j-.5)*.36, values[group]*100, width=.36,
                        label=['Presenti nel training','Assenti dal training'][j])
        axis.bar_label(bars,fmt='%.1f',fontsize=8,padding=2)
    axis.set(xticks=np.arange(6),xticklabels=short_labels,ylim=(0,110),ylabel='Percentuale dei casi nel gruppo',title=title)
    axis.legend(fontsize=8)
fig.savefig(OUTPUT/'seen_unseen_rates.png',dpi=160,bbox_inches='tight')
plt.show()

## 5. Salvare la diagnosi

I supporti permettono di ricostruire le percentuali. Non calcoliamo F1 sulle sole
righe di una classe: per questo approfondimento usiamo conteggi e recall.
Nessun parametro viene scelto sulla base di questa analisi del test.

In [ ]:
inventory.to_csv(OUTPUT/'attack_inventory.csv',index=False)
composition.to_csv(OUTPUT/'group_composition.csv',index_label='group')
group_summary.to_csv(OUTPUT/'group_metrics.csv',index=False)
family_summary.to_csv(OUTPUT/'family_group_metrics.csv',index=False)
(OUTPUT/'verification.json').write_text(json.dumps({
    'protocol_sha256': protocol_hash, 'test_rows': len(test), 'train_rows': len(train),
    'model_count': len(models), 'model_fits': 0, 'new_predictions': 0,
    'group_definition': 'raw attack name present in KDDTrain+, Normal separate',
    'input_sha256': protocol['input_sha256'],
},indent=2)+'\n',encoding='utf-8')

## Risultato della diagnosi

9.083 attacchi di tipi presenti nel training, 3.750 di tipi assenti, 9.711 Normal.
Tutti i modelli classificano peggio il gruppo assente. Per MLP: 76,33% famiglie
corrette nel gruppo presente, 39,97% in quello assente; 48,69% degli assenti viene
almeno rilevato come attacco. Non sono F1 e non misurano i falsi allarmi.
Il confronto entro famiglia e i limiti sono in docs/ATTACK_DIAGNOSIS.md.
